# 🔬 Benchmark Komparasi Arsitektur Deep Learning: Klasifikasi Jenis Sapi
### Studi Komparatif: MobileNetV3Large vs MobileNetV2 vs EfficientNetB0 vs ResNet50

Notebook ini dirancang untuk melakukan evaluasi empiris menyeluruh terhadap 4 arsitektur *Convolutional Neural Network* (CNN) berbasis *Transfer Learning* untuk klasifikasi citra jenis sapi.

#### ⚙️ Panduan Penggunaan di Google Colab (Free Tier):
1. **Pilih Akselerator GPU**:
   - Buka menu **Runtime** > **Change runtime type** (Ubah jenis runtime).
   - Pilih **Hardware accelerator**: **T4 GPU** (Tersedia gratis di Colab).
   - Klik **Save**.
2. **Optimasi Memori (Anti-OOM)**:
   - Notebook ini telah dilengkapi pembersihan *cache* memori GPU otomatis (`tf.keras.backend.clear_session()` & `gc.collect()`) setelah setiap model selesai dilatih, sehingga aman dijalankan secara beruntun di RAM gratis Colab (~12-15 GB).
3. **Hasil Akhir**:
   - Menghasilkan metrik klasifikasi lengkap (Accuracy, Precision, Recall, Macro F1, Confusion Matrix).
   - Menghasilkan metrik efisiensi deployment (Jumlah Parameter, Ukuran Berkas Keras & TFLite, serta Latensi Inferensi ms).
   - Menghasilkan berkas tabel `benchmark_results.csv` yang siap dikutip di naskah publikasi ilmiah/laporan penelitian.


In [ ]:
# 1. Verifikasi Lingkungan & Akselerasi GPU
import sys
import os

# Install dependensi yang dibutuhkan jika belum ada
!pip install -q roboflow tensorflow scikit-learn seaborn matplotlib pandas

import tensorflow as tf
print(f"📦 TensorFlow Version: {tf.__version__}")

# Deteksi ketersediaan GPU
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"🚀 Akselerator GPU Terdeteksi: {gpus[0].name}")
    try:
        # Mengaktifkan alokasi memori dinamis (memory growth) agar tidak memonopoli VRAM
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("✅ GPU Memory Growth berhasil diaktifkan.")
    except RuntimeError as e:
        print(f"Info: {e}")
else:
    print("⚠️ GPU tidak terdeteksi. Pelatihan akan berjalan di CPU (disarankan beralih ke T4 GPU di menu Runtime).")


In [ ]:
# 2. Import Libraries & Reproducibility Setup
import gc
import json
import time
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score, precision_score, recall_score

# TensorFlow & Keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, CSVLogger

# Keras Applications
from tensorflow.keras.applications import (
    MobileNetV3Large,
    MobileNetV2,
    EfficientNetB0,
    ResNet50
)
import tensorflow.keras.applications.mobilenet_v3 as mobilenet_v3
import tensorflow.keras.applications.mobilenet_v2 as mobilenet_v2
import tensorflow.keras.applications.efficientnet as efficientnet
import tensorflow.keras.applications.resnet50 as resnet50

# Set seed agar hasil eksperimen dapat direproduksi
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("✅ Semua pustaka berhasil dimuat.")


## 📂 Pemuatan & Penyiapan Dataset
Dataset citra sapi diunduh secara langsung dari Roboflow (atau dapat diarahkan ke folder lokal / Google Drive Anda).
Dataset ini memuat citra jenis sapi yang telah dipisahkan ke dalam folder `train`, `valid`, dan `test`.


In [ ]:
# 3. Download / Load Dataset
# Silakan sesuaikan API Key atau gunakan path lokal jika data sudah tersimpan di Drive
ROBOFLOW_API_KEY = "YOUR_ROBOFLOW_API_KEY_HERE"  # API key dataset Sapi Lokal
WORKSPACE_NAME = "valll-rtclj"
PROJECT_NAME = "sapi-lokal"
DATASET_VERSION = 1

# Opsi: Gunakan True jika ingin mengunduh dari Roboflow, False jika memakai folder lokal
DOWNLOAD_FROM_ROBOFLOW = True
LOCAL_DATASET_DIR = "/content/sapi-lokal-1"

if DOWNLOAD_FROM_ROBOFLOW:
    from roboflow import Roboflow
    print("📥 Mengunduh dataset dari Roboflow...")
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    project = rf.workspace(WORKSPACE_NAME).project(PROJECT_NAME)
    version = project.version(DATASET_VERSION)
    dataset = version.download("folder")
    DATASET_DIR = dataset.location
else:
    DATASET_DIR = LOCAL_DATASET_DIR

TRAIN_DIR = os.path.join(DATASET_DIR, "train")
VALID_DIR = os.path.join(DATASET_DIR, "valid")
TEST_DIR = os.path.join(DATASET_DIR, "test")

# Jika folder test tidak tersedia dari split Roboflow, gunakan valid sebagai test
if not os.path.exists(TEST_DIR) or len(os.listdir(TEST_DIR)) == 0:
    print("ℹ️ Folder test terpisah tidak ditemukan, menggunakan 'valid' untuk evaluasi.")
    TEST_DIR = VALID_DIR

print(f"📁 Path Dataset: {DATASET_DIR}")
print(f"  ├─ Train Dir: {TRAIN_DIR}")
print(f"  ├─ Valid Dir: {VALID_DIR}")
print(f"  └─ Test Dir : {TEST_DIR}")

# Periksa kelas
classes = sorted([d for d in os.listdir(TRAIN_DIR) if os.path.isdir(os.path.join(TRAIN_DIR, d))])
NUM_CLASSES = len(classes)
print(f"\n🎯 Ditemukan {NUM_CLASSES} kelas jenis sapi: {classes}")


## 🔄 Preprocessing Pipeline & Data Generators
Setiap arsitektur CNN memerlukan normalisasi tensor input yang spesifik:
- **MobileNetV3**: Membutuhkan normalisasi rentang `[-1, 1]`.
- **MobileNetV2**: Membutuhkan normalisasi rentang `[-1, 1]`.
- **EfficientNetB0**: Membutuhkan input RGB asli rentang `[0, 255]` (normalisasi terjadi di dalam layer internal).
- **ResNet50**: Membutuhkan normalisasi gaya Caffe (*zero-centered ImageNet mean*).

Fungsi di bawah ini menyiapkan data generator yang disesuaikan secara presisi per arsitektur untuk menjamin keadilan evaluasi (*fair comparison*).


In [ ]:
# 4. Fungsi Pembuat Data Generator Khusus Tiap Arsitektur
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32

def get_data_generators(preprocess_fn):
    """
    Membuat Train, Validation, dan Test generators dengan fungsi preprocessing
    yang sesuai spesifikasi arsitektur masing-masing model.
    """
    # Data augmentasi pada data latih
    train_datagen = ImageDataGenerator(
        preprocessing_function=preprocess_fn,
        rotation_range=20,
        width_shift_range=0.1,
        height_shift_range=0.1,
        shear_range=0.1,
        zoom_range=0.15,
        horizontal_flip=True,
        fill_mode='nearest'
    )

    # Tanpa augmentasi untuk validation dan test
    val_test_datagen = ImageDataGenerator(
        preprocessing_function=preprocess_fn
    )

    train_gen = train_datagen.flow_from_directory(
        TRAIN_DIR,
        target_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        shuffle=True,
        seed=SEED
    )

    val_gen = val_test_datagen.flow_from_directory(
        VALID_DIR,
        target_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        shuffle=False
    )

    test_gen = val_test_datagen.flow_from_directory(
        TEST_DIR,
        target_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        shuffle=False
    )

    return train_gen, val_gen, test_gen

print("✅ Generator factory siap digunakan.")


## 🏗️ Definisi Arsitektur Model (Model Factory)
Seluruh 4 arsitektur diinisialisasi dengan bobot pra-latih (*pre-trained weights*) dari **ImageNet-1K**.
Setiap model dipasangkan *top classification head* yang identik:
`GlobalAveragePooling2D` ➔ `Dense(128, ReLU)` ➔ `Dropout(0.2)` ➔ `Dense(NUM_CLASSES, Softmax)`.


In [ ]:
# 5. Definisi Arsitektur Model Pembanding
MODEL_CONFIGS = {
    "MobileNetV3Large": {
        "base_fn": MobileNetV3Large,
        "preprocess_fn": mobilenet_v3.preprocess_input,
        "desc": "Baseline SapiKenal (Hard-Swish, SE blocks, NAS-optimized)"
    },
    "MobileNetV2": {
        "base_fn": MobileNetV2,
        "preprocess_fn": mobilenet_v2.preprocess_input,
        "desc": "Lightweight Mobile Klasik (Inverted residuals, linear bottlenecks)"
    },
    "EfficientNetB0": {
        "base_fn": EfficientNetB0,
        "preprocess_fn": efficientnet.preprocess_input,
        "desc": "Compound Scaling (Efisiensi parameter tinggi, top benchmark)"
    },
    "ResNet50": {
        "base_fn": ResNet50,
        "preprocess_fn": resnet50.preprocess_input,
        "desc": "Deeper Residual Network (Skip connections, standar akademis)"
    }
}

def build_classifier_model(model_name, num_classes, input_shape=(224, 224, 3)):
    """Membangun model Transfer Learning dengan classification head standar."""
    config = MODEL_CONFIGS[model_name]
    base_model = config["base_fn"](
        weights="imagenet",
        include_top=False,
        input_shape=input_shape
    )
    # Freeze backbone untuk transfer learning
    base_model.trainable = False

    x = base_model.output
    x = GlobalAveragePooling2D(name="global_avg_pool")(x)
    x = Dense(128, activation="relu", name="dense_128")(x)
    x = Dropout(0.2, name="dropout_02")(x)
    outputs = Dense(num_classes, activation="softmax", name="output_layer")(x)

    model = Model(inputs=base_model.input, outputs=outputs, name=model_name)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

# Cetak ringkasan parameter dasar masing-masing model
print("📊 Ringkasan Kapasitas Tiap Arsitektur:")
for name in MODEL_CONFIGS:
    m = build_classifier_model(name, NUM_CLASSES)
    total_p = m.count_params()
    train_p = sum(tf.keras.backend.count_params(w) for w in m.trainable_weights)
    print(f"  • {name:16}: {total_p/1e6:6.2f}M params (Trainable: {train_p/1e3:6.1f}K params) - {MODEL_CONFIGS[name]['desc']}")
    del m
    tf.keras.backend.clear_session()
    gc.collect()


## 🚀 Loop Pelatihan Otomatis (Colab Free Tier Optimized)
Pelatihan dilakukan berurutan untuk keempat model dengan:
- **Epochs**: 30 (dengan *Early Stopping* jika `val_loss` tidak membaik dalam 6 epoch berturut-turut).
- **ReduceLROnPlateau**: Menurunkan *learning rate* jika performa stagnan dalam 3 epoch.
- **ModelCheckpoint**: Menyimpan bobot terbaik (*best validation accuracy*).
- **Manajemen Memori GPU**: `tf.keras.backend.clear_session()` dan `gc.collect()` dipanggil di setiap akhir iterasi untuk mencegah kehabisan memori GPU (OOM) pada Colab.


In [ ]:
# 6. Eksekusi Pelatihan 4 Model
EPOCHS = 30
OUTPUT_DIR = Path("/content/benchmark_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

training_histories = {}
trained_models_info = {}

for model_name, cfg in MODEL_CONFIGS.items():
    print("=" * 70)
    print(f"🔥 Memulai Pelatihan: {model_name}")
    print("=" * 70)

    # Siapkan data generator khusus model ini
    train_gen, val_gen, test_gen = get_data_generators(cfg["preprocess_fn"])

    # Bangun model
    model = build_classifier_model(model_name, NUM_CLASSES)

    # Callbacks
    checkpoint_path = str(OUTPUT_DIR / f"best_{model_name}.keras")
    callbacks = [
        ModelCheckpoint(
            checkpoint_path,
            monitor="val_accuracy",
            mode="max",
            save_best_only=True,
            verbose=1
        ),
        EarlyStopping(
            monitor="val_loss",
            patience=6,
            restore_best_weights=True,
            verbose=1
        ),
        ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.2,
            patience=3,
            min_lr=1e-6,
            verbose=1
        )
    ]

    start_time = time.time()
    history = model.fit(
        train_gen,
        epochs=EPOCHS,
        validation_data=val_gen,
        callbacks=callbacks,
        verbose=1
    )
    training_duration_sec = time.time() - start_time

    # Simpan riwayat training
    training_histories[model_name] = history.history
    trained_models_info[model_name] = {
        "checkpoint_path": checkpoint_path,
        "training_time_sec": training_duration_sec,
        "epochs_completed": len(history.history["loss"])
    }

    print(f"⏱️ Selesai {model_name} dalam {training_duration_sec:.1f} detik ({len(history.history['loss'])} epoch).")

    # Bersihkan memori GPU untuk model berikutnya
    del model, train_gen, val_gen, test_gen
    tf.keras.backend.clear_session()
    gc.collect()

print("\n🎉 Seluruh 4 model berhasil dilatih!")


## 📈 Visualisasi Kurva Konvergensi Pelatihan
Grafik di bawah ini membandingkan laju konvergensi *Accuracy* dan *Loss* (Training vs Validation) untuk keempat model.


In [ ]:
# 7. Plot Kurva Akurasi dan Loss
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
palette = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]

# Accuracy Plot
ax_train_acc = axes[0, 0]
ax_val_acc = axes[0, 1]
ax_train_loss = axes[1, 0]
ax_val_loss = axes[1, 1]

for idx, (name, hist) in enumerate(training_histories.items()):
    c = palette[idx % len(palette)]
    epochs_range = range(1, len(hist["accuracy"]) + 1)
    
    ax_train_acc.plot(epochs_range, hist["accuracy"], label=name, color=c, lw=2)
    ax_val_acc.plot(epochs_range, hist["val_accuracy"], label=name, color=c, lw=2)
    ax_train_loss.plot(epochs_range, hist["loss"], label=name, color=c, lw=2)
    ax_val_loss.plot(epochs_range, hist["val_loss"], label=name, color=c, lw=2)

ax_train_acc.set_title("Training Accuracy", fontsize=13, fontweight="bold")
ax_train_acc.set_xlabel("Epoch")
ax_train_acc.set_ylabel("Accuracy")
ax_train_acc.grid(True, alpha=0.3)
ax_train_acc.legend()

ax_val_acc.set_title("Validation Accuracy", fontsize=13, fontweight="bold")
ax_val_acc.set_xlabel("Epoch")
ax_val_acc.set_ylabel("Accuracy")
ax_val_acc.grid(True, alpha=0.3)
ax_val_acc.legend()

ax_train_loss.set_title("Training Loss", fontsize=13, fontweight="bold")
ax_train_loss.set_xlabel("Epoch")
ax_train_loss.set_ylabel("Loss")
ax_train_loss.grid(True, alpha=0.3)
ax_train_loss.legend()

ax_val_loss.set_title("Validation Loss", fontsize=13, fontweight="bold")
ax_val_loss.set_xlabel("Epoch")
ax_val_loss.set_ylabel("Loss")
ax_val_loss.grid(True, alpha=0.3)
ax_val_loss.legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "training_curves_comparison.png", dpi=200)
plt.show()


## 🎯 Evaluasi Komprehensif pada Test Set Independen
Pada tahap ini, bobot terbaik masing-masing model dimuat kembali untuk diuji terhadap *held-out test set*.
Metrik yang dievaluasi mencakup:
- **Test Accuracy**
- **Macro Precision, Recall, & F1-Score**
- **Confusion Matrix Heatmap** per jenis sapi


In [ ]:
# 8. Evaluasi Performa Klasifikasi & Confusion Matrix
evaluation_results = {}
class_names = sorted(classes)

fig, axes = plt.subplots(2, 2, figsize=(16, 14))
axes_flat = axes.flatten()

for idx, (model_name, cfg) in enumerate(MODEL_CONFIGS.items()):
    print(f"📊 Mengevaluasi: {model_name}...")
    
    # Siapkan data generator khusus model
    _, _, test_gen = get_data_generators(cfg["preprocess_fn"])
    
    # Load model dengan bobot terbaik
    best_path = trained_models_info[model_name]["checkpoint_path"]
    model = tf.keras.models.load_model(best_path)
    
    # Prediksi
    y_true = test_gen.classes
    test_gen.reset()
    y_pred_probs = model.predict(test_gen, verbose=0)
    y_pred = np.argmax(y_pred_probs, axis=1)
    
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="macro", zero_division=0)
    rec = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    
    evaluation_results[model_name] = {
        "test_accuracy": acc,
        "macro_precision": prec,
        "macro_recall": rec,
        "macro_f1": f1,
        "y_true": y_true,
        "y_pred": y_pred
    }
    
    # Plot Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    ax = axes_flat[idx]
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_title(f"{model_name}\n(Test Acc: {acc*100:.2f}% | F1: {f1:.3f})", fontsize=12, fontweight="bold")
    ax.set_xlabel("Prediksi")
    ax.set_ylabel("Aktual")
    
    del model, test_gen
    tf.keras.backend.clear_session()
    gc.collect()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confusion_matrices_comparison.png", dpi=200)
plt.show()

# Cetak perbandingan metrik klasifikasi
metrics_df = pd.DataFrame([
    {
        "Arsitektur": name,
        "Test Accuracy (%)": f"{res['test_accuracy']*100:.2f}%",
        "Macro Precision": f"{res['macro_precision']:.4f}",
        "Macro Recall": f"{res['macro_recall']:.4f}",
        "Macro F1-Score": f"{res['macro_f1']:.4f}"
    }
    for name, res in evaluation_results.items()
])
print("\n📋 Tabel Metrik Klasifikasi:")
display(metrics_df)


## 📱 Konversi ke TFLite & Profiling Deployment Mobile
Sesuai kebutuhan arsitektur mobile offline pada SapiKenal dan persyaratan efisiensi komputasi:
1. Model dikonversi ke format **TensorFlow Lite (`.tflite` FP32)**.
2. Ukuran berkas Keras vs TFLite diukur dalam Megabytes (MB).
3. Latensi inferensi diukur per citra menggunakan TFLite Interpreter (rata-rata 100 pengujian).


In [ ]:
# 9. Profiling Deployment: Konversi TFLite, Ukuran Berkas, dan Latensi
deployment_results = {}
DUMMY_INPUT = np.random.rand(1, 224, 224, 3).astype(np.float32)

for model_name, info in trained_models_info.items():
    print(f"⚙️ Memproses ekspor & profiling TFLite: {model_name}...")
    keras_path = Path(info["checkpoint_path"])
    tflite_path = OUTPUT_DIR / f"{model_name}.tflite"

    # Ukuran berkas Keras
    keras_size_mb = keras_path.stat().st_size / (1024 * 1024)

    # Muat Keras model untuk konversi
    model = tf.keras.models.load_model(str(keras_path))
    total_params = model.count_params()

    # Konversi ke TFLite (Float32 standar)
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    tflite_model_bytes = converter.convert()

    with open(tflite_path, "wb") as f:
        f.write(tflite_model_bytes)
    tflite_size_mb = tflite_path.stat().st_size / (1024 * 1024)

    # Benchmark Latensi Inferensi TFLite (100 iterasi)
    interpreter = tf.lite.Interpreter(model_path=str(tflite_path))
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    # Warmup
    for _ in range(10):
        interpreter.set_tensor(input_details[0]['index'], DUMMY_INPUT)
        interpreter.invoke()

    # Pengukuran latensi
    latencies = []
    for _ in range(100):
        t0 = time.perf_counter()
        interpreter.set_tensor(input_details[0]['index'], DUMMY_INPUT)
        interpreter.invoke()
        _ = interpreter.get_tensor(output_details[0]['index'])
        latencies.append((time.perf_counter() - t0) * 1000.0)

    avg_latency_ms = float(np.mean(latencies))
    std_latency_ms = float(np.std(latencies))

    deployment_results[model_name] = {
        "total_params_m": total_params / 1e6,
        "keras_size_mb": keras_size_mb,
        "tflite_size_mb": tflite_size_mb,
        "avg_latency_ms": avg_latency_ms,
        "std_latency_ms": std_latency_ms,
        "tflite_path": str(tflite_path)
    }

    del model, interpreter
    tf.keras.backend.clear_session()
    gc.collect()

print("✅ Konversi dan profiling TFLite selesai.")


## 🏆 Tabel Master Komparasi & Visualisasi Trade-Off
Tabel di bawah ini merangkum seluruh dimensi evaluasi (Akurasi, F1, Ukuran, Parameter, Latensi, dan Waktu Pelatihan) sebagai data rujukan utama bagi publikasi ilmiah atau laporan penelitian.


In [ ]:
# 10. Penyusunan Tabel Ringkasan Akhir & Visualisasi Perbandingan
summary_rows = []

for name in MODEL_CONFIGS:
    eval_m = evaluation_results[name]
    dep_m = deployment_results[name]
    info_m = trained_models_info[name]

    summary_rows.append({
        "Model": name,
        "Test Accuracy (%)": round(eval_m["test_accuracy"] * 100, 2),
        "Macro F1-Score": round(eval_m["macro_f1"], 4),
        "Total Params (M)": round(dep_m["total_params_m"], 2),
        "Keras Size (MB)": round(dep_m["keras_size_mb"], 2),
        "TFLite Size (MB)": round(dep_m["tflite_size_mb"], 2),
        "Latency CPU (ms)": round(dep_m["avg_latency_ms"], 2),
        "Training Time (s)": round(info_m["training_time_sec"], 1)
    })

master_df = pd.DataFrame(summary_rows)

# Simpan ke CSV untuk lampiran laporan / jurnal
csv_path = OUTPUT_DIR / "benchmark_comparison_results.csv"
master_df.to_csv(csv_path, index=False)
print(f"💾 Tabel berhasil disimpan ke: {csv_path}\n")

print("=" * 95)
print("             TABEL MASTER KOMPARASI ARSITEKTUR KLASIFIKASI JENIS SAPI")
print("=" * 95)
display(master_df)

# Visualisasi Trade-off: Akurasi vs Ukuran TFLite vs Latensi
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Akurasi vs Macro F1
axes[0].bar(master_df["Model"], master_df["Test Accuracy (%)"], color="#2b5c8f", alpha=0.85, width=0.5)
axes[0].set_title("Test Accuracy (%)", fontweight="bold")
axes[0].set_ylim(0, 100)
for i, v in enumerate(master_df["Test Accuracy (%)"]):
    axes[0].text(i, v + 1.5, f"{v}%", ha='center', fontweight="bold")

# Plot 2: Ukuran TFLite (MB)
axes[1].bar(master_df["Model"], master_df["TFLite Size (MB)"], color="#e07a5f", alpha=0.85, width=0.5)
axes[1].set_title("Ukuran Model TFLite (MB)", fontweight="bold")
for i, v in enumerate(master_df["TFLite Size (MB)"]):
    axes[1].text(i, v + 0.5, f"{v} MB", ha='center', fontweight="bold")

# Plot 3: Latensi Inferensi (ms)
axes[2].bar(master_df["Model"], master_df["Latency CPU (ms)"], color="#81b29a", alpha=0.85, width=0.5)
axes[2].set_title("Latensi Inferensi TFLite (ms)", fontweight="bold")
for i, v in enumerate(master_df["Latency CPU (ms)"]):
    axes[2].text(i, v + 0.8, f"{v} ms", ha='center', fontweight="bold")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "tradeoff_comparison_chart.png", dpi=200)
plt.show()


## 📝 Panduan Analisis & Interpretasi Hasil untuk Laporan/Jurnal

Gunakan panduan berikut untuk menuliskan bab pembahasan (*Results and Discussion*) pada naskah ilmiah Anda:

1. **Trade-Off Akurasi vs. Kompleksitas Model**:
   - Bandingkan apakah **ResNet50** (~25.6M parameter) memberikan peningkatan akurasi yang signifikan dibandingkan arsitektur *lightweight* seperti **MobileNetV3Large** dan **EfficientNetB0**.
   - Jika akurasi *lightweight models* mendekati atau melampaui ResNet50, ini menjadi argumen kuat bahwa untuk tugas klasifikasi 6-7 jenis sapi, model kompak sudah sangat memadai tanpa beban komputasi berlebih.
2. **Kesesuaian dengan Aplikasi Lapangan (Mobile Deployment)**:
   - Evaluasi ukuran berkas `.tflite`. Pada perangkat mobile peternak di lapangan, model dengan ukuran di bawah 15 MB dan latensi inferensi di bawah 50 ms jauh lebih unggul dalam aspek *user experience* dan efisiensi memori RAM.
3. **Penyebab Kesalahan Klasifikasi (Confusion Matrix Insight)**:
   - Amati kelas sapi mana yang saling bertukar (*misclassified*). Umumnya persilangan sapi (misal PO vs Brahman atau Limusin cross) memiliki kemiripan warna dan bentuk punuk yang menjadi tantangan visual model.

---
*Dibuat untuk kebutuhan eksperimen perbandingan model Proyek Riset SapiKenal.*
